In [ ]:
%load_ext autoreload
%autoreload 2

# Tillicum Slurm Ops

A guide to setting up SSH, managing persistent slurm jobs, and working with tillicum remotely.

## 1. SSH Config Setup

The `ssh_config_templates/` directory contains two files that need to go into your `~/.ssh/`:

- **`config`** - Main SSH config with hosts for `klone-login`, `klone-node`, `tillicum-login`, and `moana`. Uses `ControlMaster` for persistent connections so you only authenticate once.
- **`klone-node-config`** - Included by the main config for the `klone-node` host. Contains the `ProxyJump` setup so you can SSH directly to a compute node through the login node.

**Before copying**, edit `ssh_config_templates/config` and replace `deanlcs` with your own username.

Then copy them into place:

In [ ]:
! cp ssh_config_templates/* ~/.ssh/

## 2. First Login to Tillicum

Test that your SSH config works. The first time you connect you'll need to authenticate (2FA, password, etc). After that, `ControlMaster` keeps the connection alive so subsequent SSH commands reuse it without re-authenticating.

In [ ]:
! ssh tillicum-login echo "Connected successfully"

Connected successfully


## 3. Copy the Job Controller Script

`job_controller.sh` manages a persistent interactive slurm job named `proxy_jump`. It either:
- **Starts** a new interactive job if none is running
- **Reports** the node and remaining time if one already exists

Copy it to the remote host:

In [ ]:
# ! scp job_controller.sh tillicum-login:~/

## 4. Start the Job Controller

This SSHs into tillicum, creates (or attaches to) a tmux session called `job_controller`, and gives you a bash terminal on a compute node.

- If no `proxy_jump` job exists, it starts one interactively via `srun`
- If one already exists, it prints the node and time remaining
- The tmux session persists even if you disconnect

**This requires a real terminal** (tmux needs a TTY). Run the printed command in your terminal:

In [ ]:
SRUN_ARGS = "--qos=debug --gpus=1 --cpus-per-task=8 --mem=200G --time=01:00:00"
print(f"""ssh tillicum-login -t 'tmux new-session -A -s job_controller "bash ~/job_controller.sh {SRUN_ARGS}; bash"'""")

ssh tillicum-login -t 'tmux new-session -A -s job_controller "bash ~/job_controller.sh --qos=debug --gpus=1 --cpus-per-task=8 --mem=200G --time=01:00:00; bash"'


## 5. Run Commands on the Compute Node

Use `srun --overlap --jobid=ID` to run a command inside the existing job allocation and capture the output back to the notebook. This doesn't need a TTY — it runs the command and returns the result.

In [ ]:
from slurm_ops import run_on_job

In [ ]:
# Which node is the job running on?
_ = run_on_job("hostname")

g003


In [ ]:
# Which ports are in use on the compute node?
_ = run_on_job("ss -tlnp")

State  Recv-Q Send-Q Local Address:Port  Peer Address:PortProcess
LISTEN 0      6          127.0.0.1:16000      0.0.0.0:*          
LISTEN 0      6          127.0.0.1:6666       0.0.0.0:*          
LISTEN 0      6          127.0.0.1:5555       0.0.0.0:*          
LISTEN 0      4096         0.0.0.0:6818       0.0.0.0:*          
LISTEN 0      4096         0.0.0.0:1191       0.0.0.0:*          
LISTEN 0      4096         0.0.0.0:111        0.0.0.0:*          
LISTEN 0      128          0.0.0.0:22         0.0.0.0:*          
LISTEN 0      4096               *:9306             *:*          
LISTEN 0      4096               *:9303             *:*          
LISTEN 0      4096               *:9400             *:*          
LISTEN 0      4096               *:1910             *:*          
LISTEN 0      4096            [::]:111           [::]:*          
LISTEN 0      128             [::]:22            [::]:*          


In [ ]:
result = run_on_job("""python3 -c "
import socket
for p in range(8000,9000):
 s=socket.socket()
 try: s.bind(('',p)); print(p); s.close(); break
 except OSError: s.close()
" """)
free_port = result.stdout.strip()
free_port

8000


'8000'

## 6. Check Job Status / Get a Terminal

You can check the job status without tmux, or attach interactively to get a full terminal.

In [ ]:
node = run_on_job("hostname").stdout.strip()
print(f"""sed -I '' -E s"/Hostname.*/Hostname {node}/" ~/.ssh/tillicum-node-config""")

# from here
# focus on the following flows
# spinnning an interactive process with tmux (have the loading part of the command)
# cd to a repo with vllm, load the modules and run it
# getting the node of the running job
# so we can switch the ssh config to point to it
# or we can port forward to it 

g003
sed -I '' -E s"/Hostname.*/Hostname g003/" ~/.ssh/tillicum-node-config


In [ ]:
# Check job status remotely (no tmux needed)
! ssh tillicum-login 'squeue --me --name=proxy_jump --format="%i %N %L %T" --noheader'

# ssh -O exit tillicum-login


58045 g001 59:51 RUNNING


To get an interactive terminal, run this in a regular terminal (not the notebook):

```bash
# Attach to the tmux session (gives you a full terminal on the compute node)
ssh tillicum-login -t 'tmux attach -t job_controller'
```

To detach from tmux without killing it: press `Ctrl-b` then `d`.


## 7. Port Forwarding

Forward a port from the compute node to your local machine. This looks up which node the `proxy_jump` job is on and prints the SSH command to run in your terminal.

Set `PORT` to the remote port you want to forward (e.g. 8555 for vllm). Optionally set `LOCAL_PORT` if you want a different port locally.

In [ ]:
from slurm_ops import port_forward

PORT = 8555
# sets up port forwarding and returns local port
local_port = port_forward(PORT)

In [ ]:
from openai import OpenAI

# Modify OpenAI's API key and API base to use vLLM's API server.
openai_api_key = "123"
openai_api_base = "http://localhost:8555/v1"
client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)
completion = client.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    prompt="who are you?",
)
print("Completion result:", completion)

APIConnectionError: Connection error.


## 8. Useful Commands Reference

```bash
# Cancel the running job
ssh tillicum-login 'scancel --name=proxy_jump'

# Kill the tmux session
ssh tillicum-login 'tmux kill-session -t job_controller'

# List all your running jobs
ssh tillicum-login 'squeue --me'
```

## Launch vLLM Server

`vllm_server.sh` starts a vLLM server on the compute node using the existing `proxy_jump` allocation. It's parameterized with env vars (`VLLM_MODEL`, `VLLM_PORT`, `VLLM_API_KEY`).

We run it via `run_on_job` with `nohup ... &` so it starts in the background and the cell returns immediately.

In [ ]:
# Copy the script to the remote (one-time)
! scp vllm_server.sh tillicum-login:~/

vllm_server.sh                                100%  740   158.4KB/s   00:00    


In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
PORT = 8555
API_KEY = "my-secret-key"
CHDIR = "eval-bench"

# Start vLLM in the background on the existing proxy_jump compute node
# _ = run_on_job(
#     f"CHDIR={CHDIR} VLLM_MODEL={MODEL} VLLM_PORT={PORT} VLLM_API_KEY={API_KEY} "
#     f"bash ~/vllm_server.sh > ~/vllm_server.log 2>&1 &"
# )

In [ ]:
# Check if vllm is up (look for the port listening)
_ = run_on_job(f"ss -tlnp | grep {PORT} || echo 'Not listening yet — check ~/vllm_server.log'")

LISTEN 0      2048         0.0.0.0:8555       0.0.0.0:*    users:(("vllm",pid=3511842,fd=25))           


In [ ]:
# Set up port forwarding to the vLLM server
from slurm_ops import port_forward

local_port = port_forward(PORT)

Run this command to set up port forwarding:
  ssh -O forward -L 62636:g003.hyak.uw.edu:8555 tillicum-login


In [ ]:
local_port

62636

In [ ]:
from openai import OpenAI
local_port = 62637
client = OpenAI(api_key="123", base_url=f"http://localhost:{local_port}/v1")
completion = client.completions.create(
    model=MODEL,
    prompt="who are you?",
)
print("Completion result:", completion)

Completion result: Completion(id='cmpl-b086f9558c62e30b', choices=[CompletionChoice(finish_reason='length', index=0, logprobs=None, text=" I'm an AI assistant. How can I help you today?\n\nI need a", stop_reason=None, token_ids=None, prompt_logprobs=None, prompt_token_ids=None)], created=1771288463, model='Qwen/Qwen2.5-1.5B-Instruct', object='text_completion', system_fingerprint=None, usage=CompletionUsage(completion_tokens=16, prompt_tokens=4, total_tokens=20, completion_tokens_details=None, prompt_tokens_details=None), service_tier=None, kv_transfer_params=None)


In [ ]:
# Kill the vLLM server when done
_ = run_on_job(f"pkill -f 'vllm serve.*--port {PORT}'")

## 9. First-Time Setup on a Compute Node

Once you have an interactive job running (via the job controller), attach to it and run these one-time setup steps.

### Scratch directory

Home directories have limited space. Create a scratch directory and symlink `~/.cache` to it so that tools like `uv` and `huggingface-cli` cache to the larger filesystem.

```bash

mkdir -p /gpfs/scrubbed/$USER/cache
chmod 700 /gpfs/scrubbed/$USER
ln -sf /gpfs/scrubbed/$USER/cache ~/.cache
```

### Install direnv
curl -sfL https://direnv.net/install.sh | bash

set UV 


### Install uv (Python package manager)

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

This installs to `~/.local/bin`. Make sure it's on your PATH (the installer adds it to `.bashrc` automatically). To avoid the "reduced performance" warning when cache and venv are on different filesystems:

```bash
# Add to ~/.bashrc
export UV_CACHE_DIR="/gpfs/scrubbed/$USER/cache/uv"
export UV_PYTHON_INSTALL_DIR="/gpfs/scrubbed/$USER/cache/uv/python"

# Then install a Python version once:
uv python install 3.12
```

### Install GitHub CLI

```bash
VERSION="2.86.0"
ARCH="amd64"

mkdir -p ~/.local/bin && \
cd /tmp && \
wget -q "https://github.com/cli/cli/releases/download/v${VERSION}/gh_${VERSION}_linux_${ARCH}.tar.gz" -O gh.tar.gz && \
tar -xzf gh.tar.gz && \
cp gh_${VERSION}_linux_${ARCH}/bin/gh ~/.local/bin/ && \
chmod +x ~/.local/bin/gh && \
rm -rf gh_${VERSION}_linux_${ARCH} gh.tar.gz
```

Then authenticate. You'll need a [personal access token](https://github.com/settings/tokens):

```bash
gh auth login
```

### Load CUDA (needed each job)

This is not a one-time step -- you need to run this every time you start a new job:

```bash
module load gcc/13.4.0
module load cuda/13.0.0
```

Consider adding these to your `~/.bashrc` or to `job_controller.sh` if you always need them.

## Tillicum old

Get uv and hf on tillicum

```bash

# make your own scratch dir

mkdir /gpfs/scrubbed/deanlcs
mkdir /gpfs/scrubbed/deanlcs/cache
chmod 700 /gpfs/scrubbed/deanlcs

# symlink home dir cache to scratch dir
ln -s /gpfs/scrubbed/deanlcs/cache ~/.cache


# download gh
VERSION="2.86.0"  # Set your desired version
ARCH="amd64"      # or "arm64" for ARM systems

mkdir -p ~/.local/bin && \
cd /tmp && \
wget -q "https://github.com/cli/cli/releases/download/v${VERSION}/gh_${VERSION}_linux_${ARCH}.tar.gz" -O gh.tar.gz && \
tar -xzf gh.tar.gz && \
cp gh_${VERSION}_linux_${ARCH}/bin/gh ~/.local/bin/ && \
chmod +x ~/.local/bin/gh && \
rm -rf gh_${VERSION}_linux_${ARCH} gh.tar.gz && \
echo 'export PATH="$HOME/.local/bin:$PATH"' >> ~/.bashrc && \
source ~/.bashrc && \
gh --version

# go make a personal access token here https://github.com/settings/tokens
# run auth to login
gh auth login


# clone some repo
git clone https://github.com/...


#TODO fix from here

# when starting a new job, we need to load cuda by running
salloc --qos=debug --gpus=1 --cpus-per-task=8 --mem=200G --time=01:00:00
module load gcc/13.4.0
module load cuda/13.0.0


# set the UV_CACHE and UV project env env vars in the env file the uv project is looking at
export UV_PROJECT_ENVIRONMENT="/gpfs/scrubbed/deanlcs/cache/uv/eval_bench"
export UV_CACHE_DIR="/gpfs/scrubbed/deanlcs/cache/uv"
source /gpfs/scrubbed/deanlcs/cache/uv/eval_bench/bin/activate

vllm serve Qwen/Qwen2.5-1.5B-Instruct --port 8555 --api-key 123

shh tillicum.hyak.uw.edu -L 8555:g001.hyak.local:8555

# uv "reduced performance" / different-filesystem warning:
# uv wants cache and the venv on the *same* filesystem so it can hardlink from cache into the env.
# If they're on different mounts (e.g. home vs scratch), it falls back to full copies and warns.
# Fix 1 (best): put Python install on scratch too so cache + env + Python are all same FS:
export UV_PYTHON_INSTALL_DIR="/gpfs/scrubbed/deanlcs/cache/uv/python"
# Then run once: uv python install 3.12   (or whatever version) so it installs there.
#
# Fix 2 (if you can't get same FS): tell uv to not use hardlinks; avoids the fallback surprise.
# export UV_LINK_MODE=copy    # slower but predictable
# export UV_LINK_MODE=symlink # works across FS; env breaks if cache is moved/deleted

# TODO from here, make an ondemand vscode server and setup vllm on eval-bench with it 
```

## hyak old

```bash

# login to main node
ssh klone-login

# open a tmux window to avoid detach bugs
tmux

# hyak
# request an interactive job for 4 hours
salloc --account=argon --partition=gpu-l40 --gpus=1 --mem=64G --time=4:00:00 --job-name=vsc-proxy-jump


# tillicum
salloc --qos=debug --gpus=1 --cpus-per-task=8 --mem=200G --time=01:00:00



# on a new terminal 
# change your hyak node ssh config by running
bash ~/.ssh/set-hyak-node.sh

# then ssh to klone-node
ssh klone-node

# now you can use proxyjump extension (Remote ssh to klone node)


# if you want to find the jobs you are running
squeue --user $USER

# and to cancel a job
scancel JOBID

```